# 09 Survival and person-time: start times, competing events, adjusted curves, life tables, restricted means

The survival words and options added after simplydata 0.2.0, each on real data shipped with the R packages survival
and KMsurv:

| what | simplydata | data |
|---|---|---|
| person-time from dates | `add_time_and_event_columns` | UDCA trial (survival::udca) |
| survival at chosen times | `find_survival_over_time(at_time_points=)` | NCCTG lung cancer |
| restricted mean survival time | `find_survival_over_time(average_time_up_to=)`, `compare_survival(average_time_up_to=)` | NCCTG lung cancer |
| actuarial life table | `find_survival_over_time(method="life table")` | breast feeding (KMsurv::bfeed) |
| curves adjusted for other columns | `find_survival_over_time(allowing_for=)` | NCCTG lung cancer |
| delayed entry | `find_survival_over_time(followed_from=)`, `compare_survival(followed_from=)` | Channing House (KMsurv) |
| competing events | `find_risk_over_time`, Gray's test, cause-specific Cox, Fine-Gray | mgus2 (survival) |
| stratified log-rank and Cox | `compare_survival(allowing_for=)`, `explain_survival(allowing_for=)` | veteran (survival) |
| start-stop rows | `explain_survival(followed_from=)` | Stanford heart transplant (survival::heart) |
| parametric (Weibull) model | `explain_survival(distribution="weibull")` | larynx cancer (KMsurv) |

**Only simplydata runs in this notebook.** The reference values (`references/09_survival_and_person_time.csv`, from
`scripts/references/09_survival_and_person_time.py`) are the original libraries run once: **R** survival (`survfit`, `survdiff`, `coxph`, `survreg`), cmprsk (`cuminc`, `crr`), survRM2 (`rmst2`),
KMsurv (`lifetab`) and adjustedCurves (`adjustedsurv`), run here; **Python**: statsmodels (`SurvfuncRight`,
`CumIncidenceRight`, `survdiff`, `PHReg`) and lifelines (`KaplanMeierFitter`, `CoxPHFitter`,
`CoxTimeVaryingFitter`, `WeibullAFTFitter`), and the closed formulas written out (person-time, the actuarial life
table). Every check prints its source. No published table values are used here: the textbook tables these datasets come from print results
rounded to 3 or 4 digits and were themselves made with these R packages.

Tolerances (relative difference, the same for every dataset): **1e-10** for results from closed formulas
(Kaplan-Meier and Aalen-Johansen estimates, the life table, restricted means, the log-rank and Gray's tests,
person-time) and **1e-6** for models fitted by iteration (Cox, Fine-Gray, Weibull, the adjusted curves, which rest
on a Cox model).

lifelines is run with tighter convergence than its defaults (`CoxPHFitter`: `precision=1e-12`; `WeibullAFTFitter`:
L-BFGS-B with `ftol=1e-15`, `gtol=1e-12`), because its defaults stop 1e-6 to 1e-5 short of the solution; this only
makes the comparison stricter.

The tables read here are the data files of this repository with the recoding the notebook needs (yes/no for
died, male/female for sex, ...), written by the reference script to `references/data/`.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_numbers, read_rows, versions

NOTEBOOK = "09_survival_and_person_time"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def rows(word, *args, **options):
    return read_rows(quiet(word, *args, **options))


def by_term(word, *args, **options):
    return {row["term"]: row for row in rows(word, *args, **options)}


lung = load("lung")

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Person-time from dates: `add_time_and_event_columns` on the UDCA trial

170 patients with primary biliary cirrhosis (ursodeoxycholic acid against placebo): date of entry, last follow-up
and death. Time runs from entry to the first of death and last follow-up, in years of 365.25 days. References: pandas
date arithmetic written out, and base R `difftime`.

In [2]:
udca = load("udca")
for column in ("entry.dt", "last.dt", "death.dt"):
    udca.turn_into_dates(column)
udca.add_time_and_event_columns(from_date="entry.dt", to_date="last.dt", event_date="death.dt", unit="years")

entry.dt is now dates.
last.dt is now dates.
death.dt is now dates.
Added "time": the years from "entry.dt" to the first of "death.dt" and "last.dt".
Added "event": 1 when "death.dt" came first (on or before the other dates), else 0 (11 of 170 rows).
Person-time in all: 557.665 years over 170 rows.
Note: one year is 365.25 days here (as R Epi cal.yr() and Stata stset scale(365.25)); other tools may count calendar months or years.
Note: 1 row starts and ends on the same day: counted as 0, so it adds no person-time (add 1 day if your days count both ends).


,id,trt,entry.dt,last.dt,stage,bili,riskscore,death.dt,tx.dt,hprogress.dt,varices.dt,ascites.dt,enceph.dt,double.dt,worsen.dt,time,event
1,1,1,1988-04-21,1993-06-30,1,1.0,5.1,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.190965,0
2,2,0,1988-04-27,1993-06-30,1,1.7,4.2,NaT,NaN,NaN,NaN,NaN,NaN,NaN,1992-04-22,5.174538,0
3,3,1,1988-04-25,1993-06-30,0,0.5,3.4,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.180014,0
4,4,1,1988-04-27,1989-04-05,1,1.4,5.0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.939083,0
5,5,0,1988-05-12,1993-06-30,1,1.1,4.3,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.133470,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,166,0,1991-02-13,1993-06-30,1,0.5,4.4,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.376454,0
167,167,1,1991-03-07,1993-06-30,0,0.7,3.8,NaT,NaN,NaN,NaN,NaN,NaN,NaN,1993-05-25,2.316222,0
168,168,1,1991-03-28,1993-06-30,1,0.9,4.0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.258727,0
169,169,0,1991-04-19,1993-06-30,0,1.0,5.1,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.198494,0


In [3]:
time = udca.get_column("time")
event = udca.get_column("event")
ds, method = "UDCA trial", "add_time_and_event_columns"
checks.check(ds, method, "person-years", quiet(time.find_total))
checks.check(ds, method, "deaths", quiet(event.find_total))
checks.check(ds, method, "rows", time.count() - time.count_empty_numbers())
checks.check(ds, method, "time of each row (years)", read_numbers(time))

agrees  UDCA trial | add_time_and_event_columns | person-years: simplydata 557.664613279, pandas dates (written out) 557.664613279; relative difference 0.00e+00 (allowed 1e-10)
      Source: pandas dates (written out) (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=dates
agrees  UDCA trial | add_time_and_event_columns | person-years: simplydata 557.664613279, R difftime 557.664613279; relative difference 0.00e+00 (allowed 1e-10)
      Source: R difftime (run reference) - https://search.r-project.org/R/refmans/base/html/difftime.html
agrees  UDCA trial | add_time_and_event_columns | deaths: simplydata 11, pandas dates (written out) 11; relative difference 0.00e+00 (allowed 1e-10)
      Source: pandas dates (written out) (run reference) - https://pandas.pydata.org/pandas-docs/version/2.3.3/search.html?q=dates
agrees  UDCA trial | add_time_and_event_columns | deaths: simplydata 11, R difftime 11; relative difference 0.00e+00 (allowed 1e-10)
      Source

1

## Survival at chosen times and restricted mean survival time: NCCTG lung cancer

Kaplan-Meier survival at 180 and 365 days by sex, with the 95 % interval (R `survfit`'s default, on the log
scale), the number at risk and the events by then. References: R `summary(survfit(...), times =)`, statsmodels
`SurvfuncRight` and lifelines `KaplanMeierFitter` (estimates only: their default intervals use another scale).

The average days alive in the first year (restricted mean survival time up to 365 days) by sex: R
`summary(survfit(...), rmean = 365)` and survRM2 `rmst2` (estimate and standard error) and lifelines
`restricted_mean_survival_time`; the difference and ratio between the sexes with their intervals and p-value:
survRM2 `rmst2`.

In [4]:
lung.find_survival_over_time(time="time", event="died", for_each_group_in="sex", at_time_points=[180, 365])

The event is "died" = "yes" (165 of 228 rows); the other rows count as followed without the event until their "time" (censored).
Survival (the share still without the event) at the times asked, with its 95 % CI:
  female, at "time" 180: 84.2 % still without the event (95 % CI 77.0 % to 92.2 %; 71 still followed).
  female, at "time" 365: 52.6 % still without the event (95 % CI 42.1 % to 65.8 %; 30 still followed).
  male, at "time" 180: 64.4 % still without the event (95 % CI 56.9 % to 73.0 %; 89 still followed).
  male, at "time" 365: 33.6 % still without the event (95 % CI 26.1 % to 43.3 %; 35 still followed).
Method: Kaplan-Meier estimate read at those times, with 95 % confidence intervals on the log scale (Greenwood), as R summary(survfit(Surv(time, died == "yes") ~ sex), times = c(180, 365), extend = TRUE).


,sex,time,at risk,events by then,survival,95 % CI low,95 % CI high
1,female,180.0,71,14,0.842402,0.769900,0.921731
2,female,365.0,30,36,0.526463,0.421486,0.657586
3,male,180.0,89,49,0.644465,0.569284,0.729574
4,male,365.0,35,85,0.336088,0.260901,0.432943


In [5]:
found = {(row["sex"], row["time"]): row
         for row in rows(lung.find_survival_over_time, time="time", event="died", for_each_group_in="sex",
                         at_time_points=[180, 365])}
for sex in ("female", "male"):
    for t in (180, 365):
        row = found[(sex, t)]
        checks.check_values("NCCTG lung", "find_survival_over_time (at_time_points)",
                            {f"{sex} at {t}: {column}": row[column]
                             for column in ("survival", "95 % CI low", "95 % CI high", "at risk", "events by then")})

agrees  NCCTG lung | find_survival_over_time (at_time_points) | female at 180: survival: simplydata 0.842401705588, statsmodels SurvfuncRight 0.842401705588; relative difference 1.32e-16 (allowed 1e-10)
      Source: statsmodels SurvfuncRight (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=SurvfuncRight
agrees  NCCTG lung | find_survival_over_time (at_time_points) | female at 180: survival: simplydata 0.842401705588, lifelines KaplanMeierFitter 0.842401705588; relative difference 1.32e-16 (allowed 1e-10)
      Source: lifelines KaplanMeierFitter (run reference) - https://lifelines.readthedocs.io/en/v0.30.3/search.html?q=KaplanMeierFitter
agrees  NCCTG lung | find_survival_over_time (at_time_points) | female at 180: survival: simplydata 0.842401705588, R survfit 0.842401705588; relative difference 1.32e-16 (allowed 1e-10)
      Source: R survfit (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survfit.html (version 3.8.12)
agrees  NCCTG lung 

In [6]:
average = {row["sex"]: row for row in rows(lung.find_survival_over_time, time="time", event="died",
                                           for_each_group_in="sex", average_time_up_to=365)}
for sex in ("female", "male"):
    checks.check_values("NCCTG lung", "find_survival_over_time (average_time_up_to)",
                        {f"{sex}: average days up to 365": average[sex]["average time without the event"],
                         f"{sex}: standard error": average[sex]["standard error"]})
row = rows(lung.compare_survival, time="time", event="died", for_each_group_in="sex", average_time_up_to=365)[0]
checks.check_values("NCCTG lung", "compare_survival (average_time_up_to)",
                    {"difference": row["difference in average time"], "difference 95 % CI low": row["difference 95 % CI low"],
                     "difference 95 % CI high": row["difference 95 % CI high"],
                     "difference p-value": row["difference p-value"], "ratio": row["ratio of average times"],
                     "ratio 95 % CI low": row["ratio 95 % CI low"], "ratio 95 % CI high": row["ratio 95 % CI high"]})

agrees  NCCTG lung | find_survival_over_time (average_time_up_to) | female: average days up to 365: simplydata 297.465409532, lifelines restricted_mean_survival_time 297.465409532; relative difference 1.15e-15 (allowed 1e-10)
      Source: lifelines restricted_mean_survival_time (run reference) - https://lifelines.readthedocs.io/en/v0.30.3/search.html?q=restricted_mean_survival_time
agrees  NCCTG lung | find_survival_over_time (average_time_up_to) | female: average days up to 365: simplydata 297.465409532, R survfit (rmean) 297.465409532; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survfit (rmean) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survfit.html (version 3.8.12)
agrees  NCCTG lung | find_survival_over_time (average_time_up_to) | female: average days up to 365: simplydata 297.465409532, R survRM2 rmst2 297.465409532; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survRM2 rmst2 (run reference) - https://search.r-project

7

## The actuarial life table: breast feeding (KMsurv::bfeed)

927 first-born children: weeks of breast feeding (Klein & Moeschberger 2003, section 5.4), with the intervals of
their example (0, 2, 3, 5, 7, 11, 17, 25, 37, 53 weeks). References: R `KMsurv::lifetab` on the counts in each
interval, and the actuarial formulas written out in Python: effective number at risk n' = n - lost / 2,
survival at the start of each interval the product of (1 - events / n'), its standard error by Greenwood's
formula, the hazard events / (width x (n' - events / 2)) and its standard error.

In [7]:
bfeed = load("bfeed")
CUTS = [0, 2, 3, 5, 7, 11, 17, 25, 37, 53]
bfeed.find_survival_over_time(time="duration", event="delta", method="life table", at_time_points=CUTS)

Note: "method=" is now called "test_method=". The old name will stop working in a later version.
The event is "delta" = 1 (892 of 927 rows); the other rows count as followed without the event until their "duration" (censored).
Actuarial life table: rows lost in an interval count as followed for half of it (the actuarial assumption), so "effective at risk" is the number at the start minus half the lost rows.
    0 to 2: 927 at the start, 2 lost, 77 with the event; effective at risk 926; survival at the start 1.00.
    2 to 3: 848 at the start, 3 lost, 71 with the event; effective at risk 846; survival at the start 0.917.
    3 to 5: 774 at the start, 6 lost, 119 with the event; effective at risk 771; survival at the start 0.840.
    5 to 7: 649 at the start, 9 lost, 75 with the event; effective at risk 644; survival at the start 0.710.
    7 to 11: 565 at the start, 7 lost, 109 with the event; effective at risk 562; survival at the start 0.628.
    11 to 17: 449 at the start, 5 lost, 14

,interval start,interval end,at start,lost,events,effective at risk,survival at the start,standard error of survival,hazard,standard error of hazard
1,0.0,2.0,927,2,77,926.0,1.000000,0.000000,0.043380,0.004939
2,2.0,3.0,848,3,71,846.5,0.916847,0.009074,0.087546,0.010380
3,3.0,5.0,774,6,119,771.0,0.839946,0.012058,0.083626,0.007639
4,5.0,7.0,649,9,75,644.5,0.710305,0.014947,0.061779,0.007120
5,7.0,11.0,565,7,109,561.5,0.627647,0.015967,0.053748,0.005118
6,11.0,17.0,449,5,148,446.5,0.505806,0.016593,0.066219,0.005335
7,17.0,25.0,296,3,107,294.5,0.338148,0.015812,0.055498,0.005231
8,25.0,37.0,186,0,74,186.0,0.215290,0.013826,0.041387,0.004660
9,37.0,53.0,112,0,85,112.0,0.129637,0.011358,0.076439,0.006560
10,53.0,NaN,27,0,27,27.0,0.031252,0.005912,NaN,NaN


In [8]:
life = {row["interval start"]: row for row in rows(bfeed.find_survival_over_time, time="duration", event="delta",
                                                    method="life table", at_time_points=CUTS)}
QUANTITIES = ["at start", "lost", "events", "effective at risk", "survival at the start", "standard error of survival",
              "hazard", "standard error of hazard"]
for start in CUTS:
    # check_values compares only the quantities that have a reference (the last interval has no hazard)
    checks.check_values("bfeed (KMsurv)", "find_survival_over_time (life table)",
                        {f"from week {start}: {quantity}": life[start][quantity] for quantity in QUANTITIES})

agrees  bfeed (KMsurv) | find_survival_over_time (life table) | from week 0: at start: simplydata 927, actuarial formulas (Python) 927; relative difference 0.00e+00 (allowed 1e-10)
      Source: actuarial formulas (Python) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/09_survival_and_person_time.py
agrees  bfeed (KMsurv) | find_survival_over_time (life table) | from week 0: at start: simplydata 927, R KMsurv lifetab 927; relative difference 0.00e+00 (allowed 1e-10)
      Source: R KMsurv lifetab (run reference) - https://search.r-project.org/CRAN/refmans/KMsurv/html/lifetab.html (version 0.1.6)
agrees  bfeed (KMsurv) | find_survival_over_time (life table) | from week 0: lost: simplydata 2, actuarial formulas (Python) 2; relative difference 0.00e+00 (allowed 1e-10)
      Source: actuarial formulas (Python) (run reference) - https://github.com/bioforkgithub/simplydata_tests/blob/main/scripts/references/09_survival_and_person_time.py
agre

## Curves adjusted for other columns: NCCTG lung cancer, by sex, allowing for age and ECOG score

Direct adjustment (the average of every patient's predicted curve with sex set to each value in turn) from a Cox
model, as simplydata states it does: R `adjustedCurves::adjustedsurv(method = "direct")` on `coxph(Surv(time,
event) ~ group + age + ecog)`, rows with an empty ECOG score left out. A model fitted by iteration: 1e-6.

In [9]:
adjusted = rows(lung.find_survival_over_time, time="time", event="died", for_each_group_in="sex",
                allowing_for=["age", "ph.ecog"], at_time_points=[180, 365])
for row in adjusted:
    checks.check_values("NCCTG lung", "find_survival_over_time (allowing_for)",
                        {f"{row['sex']} at {row['time']:g}: {column}": row[column]
                         for column in ("survival", "95 % CI low", "95 % CI high")})

agrees  NCCTG lung | find_survival_over_time (allowing_for) | female at 180: survival: simplydata 0.795020269457, R adjustedCurves adjustedsurv 0.795020269457; relative difference 1.40e-16 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (run reference) - https://search.r-project.org/CRAN/refmans/adjustedCurves/html/adjustedsurv.html (version 0.12.0)
agrees  NCCTG lung | find_survival_over_time (allowing_for) | female at 180: 95 % CI low: simplydata 0.738088317641, R adjustedCurves adjustedsurv 0.738088317641; relative difference 0.00e+00 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (run reference) - https://search.r-project.org/CRAN/refmans/adjustedCurves/html/adjustedsurv.html (version 0.12.0)
agrees  NCCTG lung | find_survival_over_time (allowing_for) | female at 180: 95 % CI high: simplydata 0.851952221272, R adjustedCurves adjustedsurv 0.851952221272; relative difference 1.30e-16 (allowed 1e-06)
      Source: R adjustedCurves adjustedsurv (run reference) 

## Delayed entry: Channing House (KMsurv::channing)

462 residents of a retirement centre, followed from their age at entry (months) to death or the end of the study;
4 whose age at exit is not after their age at entry are left out (as in the KMsurv example). Survival to 80 years
(960 months) given alive at entry: R `survfit(Surv(ageentry, age, death) ~ 1)`, statsmodels
`SurvfuncRight(entry=)` and lifelines `KaplanMeierFitter(entry=)`. Women against men with start times: statsmodels
`survdiff(entry=)`, and the score test of R `coxph(Surv(ageentry, age, death) ~ gender)` (R `survdiff` takes no start
times; the log-rank statistic is the Cox score test, with ties handled by the exact partial likelihood).

In [10]:
channing = load("channing")
row = rows(channing.find_survival_over_time, time="age", event="death", followed_from="ageentry", at_time_points=[960])[0]
checks.check_values("Channing House", "find_survival_over_time (followed_from)",
                    {"survival at 960 months": row["survival"], "95 % CI low": row["95 % CI low"],
                     "95 % CI high": row["95 % CI high"], "at risk": row["at risk"]})
row = rows(channing.compare_survival, time="age", event="death", for_each_group_in="gender", followed_from="ageentry")[0]
checks.check_values("Channing House", "compare_survival (followed_from)",
                    {"chi-square": row["statistic"], "p-value": row["p-value"]})

agrees  Channing House | find_survival_over_time (followed_from) | survival at 960 months: simplydata 0.565869628031, statsmodels SurvfuncRight 0.565869628031; relative difference 0.00e+00 (allowed 1e-10)
      Source: statsmodels SurvfuncRight (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=SurvfuncRight
agrees  Channing House | find_survival_over_time (followed_from) | survival at 960 months: simplydata 0.565869628031, lifelines KaplanMeierFitter 0.565869628031; relative difference 1.18e-15 (allowed 1e-10)
      Source: lifelines KaplanMeierFitter (run reference) - https://lifelines.readthedocs.io/en/v0.30.3/search.html?q=KaplanMeierFitter
agrees  Channing House | find_survival_over_time (followed_from) | survival at 960 months: simplydata 0.565869628031, R survfit 0.565869628031; relative difference 0.00e+00 (allowed 1e-10)
      Source: R survfit (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survfit.html (version 3.8.12)
agrees  Chann

3

## Competing events: plasma cell malignancy or death first (survival::mgus2)

1384 people with monoclonal gammopathy (MGUS). The time is to progression to a plasma cell malignancy (PCM) when it
came first, else to death or last follow-up (Therneau's `compete` vignette). The chance of PCM by 10 years (120
months), death competing (Aalen-Johansen): R `survfit(Surv(etime, state) ~ sex)` (multi-state) and statsmodels
`CumIncidenceRight`. Gray's test between women and men: R `cmprsk::cuminc`. Cause-specific Cox model of PCM on
age and sex: R `coxph`, statsmodels `PHReg` and lifelines `CoxPHFitter` (all with Efron's ties). Fine-Gray
subdistribution model: R `cmprsk::crr`.

In [11]:
mgus = load("mgus")
mgus.find_risk_over_time(time="etime", event="outcome", value="pcm", competing_with="death", for_each_group_in="sex",
                         at_time_points=[120])

The events in "outcome": "pcm" (115 rows) and "death" (860 rows); "pcm" is the event asked about, "death" competes with it (can come first).
Counted as still followed without these events (censored) in "outcome": "censored": 409 rows.
The chance of having had each event by each time (Aalen-Johansen), with its 95 % CI:
  F, by "etime" 120: 7.4 % had had "pcm" (95 % CI 5.6 % to 9.8 %), and 48.0 % had had "death" first.
  F, if "death" were wrongly counted as the end of follow-up (1 minus Kaplan-Meier), the chance of "pcm" by "etime" 120 would come out as 10.3 %: too high, because rows that had "death" first are treated as if they could still have "pcm".
  M, by "etime" 120: 5.5 % had had "pcm" (95 % CI 4.1 % to 7.5 %), and 57.5 % had had "death" first.
  M, if "death" were wrongly counted as the end of follow-up (1 minus Kaplan-Meier), the chance of "pcm" by "etime" 120 would come out as 8.8 %: too high, because rows that had "death" first are treated as if they could still have "pcm".
M

,sex,event,time,at risk,events by then,risk,95 % CI low,95 % CI high
1,F,pcm,120.0,214,44,0.073886,0.055525,0.098318
2,F,death,120.0,214,287,0.480490,0.441396,0.523047
3,M,pcm,120.0,210,39,0.055310,0.040717,0.075134
4,M,death,120.0,210,411,0.575178,0.539232,0.613521


In [12]:
risk = rows(mgus.find_risk_over_time, time="etime", event="outcome", value="pcm", competing_with="death",
            for_each_group_in="sex", at_time_points=[120])
for row in risk:
    checks.check_values("mgus2", "find_risk_over_time (competing_with)",
                        {f"{row['sex']}, {row['event']} by 120 months: {column}": row[column]
                         for column in ("risk", "95 % CI low", "95 % CI high")})
gray = rows(mgus.compare_survival, time="etime", event="outcome", value="pcm", competing_with="death",
            for_each_group_in="sex")[0]
checks.check_values("mgus2", "compare_survival (Gray's test)", {"chi-square": gray["statistic"], "p-value": gray["p-value"]})
cause_specific = by_term(mgus.explain_survival, time="etime", event="outcome", value="pcm", competing_with="death",
                         using=["age", "sex"])
fine_gray = by_term(mgus.explain_survival, time="etime", event="outcome", value="pcm", competing_with="death",
                    using=["age", "sex"], method="fine-gray")
for term in ["age", "sex: M vs F"]:
    for method, model in (("explain_survival (competing_with, cause-specific)", cause_specific),
                          ("explain_survival (fine-gray)", fine_gray)):
        checks.check_values("mgus2", method, {f"{term}: coefficient": model[term]["coefficient"],
                                              f"{term}: standard error": model[term]["standard error"]})

agrees  mgus2 | find_risk_over_time (competing_with) | F, pcm by 120 months: risk: simplydata 0.0738856643759, statsmodels CumIncidenceRight 0.0738856643759; relative difference 1.31e-15 (allowed 1e-10)
      Source: statsmodels CumIncidenceRight (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=CumIncidenceRight
agrees  mgus2 | find_risk_over_time (competing_with) | F, pcm by 120 months: risk: simplydata 0.0738856643759, R survfit (Aalen-Johansen) 0.0738856643759; relative difference 1.31e-15 (allowed 1e-10)
      Source: R survfit (Aalen-Johansen) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survfit.html (version 3.8.12)
agrees  mgus2 | find_risk_over_time (competing_with) | F, pcm by 120 months: 95 % CI low: simplydata 0.0555247328325, R survfit (Aalen-Johansen) 0.0555247328325; relative difference 1.50e-15 (allowed 1e-10)
      Source: R survfit (Aalen-Johansen) (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/

agrees  mgus2 | explain_survival (competing_with, cause-specific) | age: coefficient: simplydata 0.0130385698371, statsmodels PHReg 0.0130385698371; relative difference 3.06e-15 (allowed 1e-06)
      Source: statsmodels PHReg (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=PHReg
agrees  mgus2 | explain_survival (competing_with, cause-specific) | age: coefficient: simplydata 0.0130385698371, lifelines CoxPHFitter 0.0130385698371; relative difference 1.26e-13 (allowed 1e-06)
      Source: lifelines CoxPHFitter (run reference) - https://lifelines.readthedocs.io/en/v0.30.3/search.html?q=CoxPHFitter
agrees  mgus2 | explain_survival (competing_with, cause-specific) | age: coefficient: simplydata 0.0130385698371, R coxph 0.013038569825; relative difference 9.29e-10 (allowed 1e-06)
      Source: R coxph (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/coxph.html (version 3.8.12)
agrees  mgus2 | explain_survival (competing_with, cause-specific) | age

## Stratified log-rank test and Cox model: veteran lung cancer trial (survival::veteran)

137 men with advanced lung cancer, standard or test chemotherapy, compared within each of the four cell types
(`allowing_for="celltype"`). Log-rank: R `survdiff(... + strata(celltype))` and statsmodels `survdiff(strata=)`.
Cox on treatment and Karnofsky score, each cell type with its own baseline: R `coxph(... + strata(celltype))`,
statsmodels `PHReg(strata=)` and lifelines `CoxPHFitter(strata=)`.

In [13]:
veteran = load("veteran")
row = rows(veteran.compare_survival, time="time", event="status", for_each_group_in="trt", allowing_for="celltype")[0]
checks.check_values("veteran", "compare_survival (allowing_for)", {"chi-square": row["statistic"], "p-value": row["p-value"]})
cox = by_term(veteran.explain_survival, time="time", event="status", using=["trt", "karno"], allowing_for="celltype")
for term in ["trt: test vs standard", "karno"]:
    checks.check_values("veteran", "explain_survival (allowing_for)", {f"{term}: coefficient": cox[term]["coefficient"],
                                                                       f"{term}: standard error": cox[term]["standard error"]})

agrees  veteran | compare_survival (allowing_for) | chi-square: simplydata 0.701743346844, statsmodels survdiff 0.701743346844; relative difference 0.00e+00 (allowed 1e-10)
      Source: statsmodels survdiff (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=survdiff
agrees  veteran | compare_survival (allowing_for) | chi-square: simplydata 0.701743346844, R survdiff 0.701743346844; relative difference 5.85e-15 (allowed 1e-10)
      Source: R survdiff (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/survdiff.html (version 3.8.12)
agrees  veteran | compare_survival (allowing_for) | p-value: simplydata 0.402198523781, statsmodels survdiff 0.402198523781; relative difference 1.38e-16 (allowed 1e-10)
      Source: statsmodels survdiff (run reference) - https://www.statsmodels.org/v0.15.0/search.html?q=survdiff
agrees  veteran | compare_survival (allowing_for) | p-value: simplydata 0.402198523781, R survdiff 0.402198523781; relative difference 3.45e

## Start-stop rows and a Weibull model

- **Stanford heart transplant** (survival::heart, 172 rows of 103 patients; a patient's second row starts at the
  transplant): Cox model on age, prior surgery and transplant with start times (`followed_from="start"`). R
  `coxph(Surv(start, stop, event) ~ age + surgery + transplant)` and lifelines `CoxTimeVaryingFitter` (Efron's
  ties in both).
- **Larynx cancer** (KMsurv::larynx, 90 patients): a Weibull accelerated failure time model on stage and age
  (`distribution="weibull"`); R `survreg(dist = "weibull")` and lifelines `WeibullAFTFitter` (its coefficients of
  lambda are survreg's coefficients).

In [14]:
heart = load("heart")
model = by_term(heart.explain_survival, time="stop", event="event", using=["age", "surgery", "transplant"],
                followed_from="start")
for term in ["age", "surgery", "transplant"]:
    checks.check_values("Stanford heart", "explain_survival (followed_from)",
                        {f"{term}: coefficient": model[term]["coefficient"],
                         f"{term}: standard error": model[term]["standard error"]})
larynx = load("larynx")
model = by_term(larynx.explain_survival, time="time", event="delta", using=["stage", "age"], distribution="weibull")
for term in ["stage: stage 2 vs stage 1", "stage: stage 3 vs stage 1", "stage: stage 4 vs stage 1", "age"]:
    checks.check_values("larynx (KMsurv)", "explain_survival (distribution weibull)",
                        {f"{term}: coefficient": model[term]["coefficient"],
                         f"{term}: standard error": model[term]["standard error"]})

agrees  Stanford heart | explain_survival (followed_from) | age: coefficient: simplydata 0.0305363149142, lifelines CoxTimeVaryingFitter 0.0305363149142; relative difference 5.81e-14 (allowed 1e-06)
      Source: lifelines CoxTimeVaryingFitter (run reference) - https://lifelines.readthedocs.io/en/v0.30.3/search.html?q=CoxTimeVaryingFitter
agrees  Stanford heart | explain_survival (followed_from) | age: coefficient: simplydata 0.0305363149142, R coxph 0.0305363149142; relative difference 9.09e-16 (allowed 1e-06)
      Source: R coxph (run reference) - https://search.r-project.org/CRAN/refmans/survival/html/coxph.html (version 3.8.12)
agrees  Stanford heart | explain_survival (followed_from) | age: standard error: simplydata 0.0138927866309, lifelines CoxTimeVaryingFitter 0.0138927867535; relative difference 8.83e-09 (allowed 1e-06)
      Source: lifelines CoxTimeVaryingFitter (run reference) - https://lifelines.readthedocs.io/en/v0.30.3/search.html?q=CoxTimeVaryingFitter
agrees  Stanfor

## Result

In [15]:
results = checks.finish()

09_survival_and_person_time: 303 comparisons - 303 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
